In [1]:
# # RUN THIS FIRST
# import os
# os.kill(os.getpid(), 9)  # Force restart

In [2]:
!pip install -q sentence-transformers==2.2.2
!pip uninstall -y qdrant-client
!pip install -q qdrant-client==1.7.0

Found existing installation: qdrant-client 1.18.0
Uninstalling qdrant-client-1.18.0:
  Successfully uninstalled qdrant-client-1.18.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 458.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
blobfile 3.2.0 requires urllib3>=2, but you have urllib3 1.26.20 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you hav

In [3]:
# FIX: Run this cell to fix the conflict, then re-run from Cell 3
import importlib
import sys

# Remove cached tree_sitter modules to clear namespace
modules_to_remove = [k for k in sys.modules.keys() if 'tree_sitter' in k]
for m in modules_to_remove:
    del sys.modules[m]

print("Cleared tree_sitter cache. Now re-run Cell 3 onwards.")

Cleared tree_sitter cache. Now re-run Cell 3 onwards.


In [4]:
# RUN THIS CELL FIRST to clear everything
import sys
# Remove all custom modules from cache
modules_to_kill = [k for k in sys.modules.keys() if 'codebase_qa' in k or '__main__' in k]
for m in modules_to_kill:
    if m in sys.modules:
        del sys.modules[m]

print("✅ Cache cleared! Now restart: Runtime → Restart session")

✅ Cache cleared! Now restart: Runtime → Restart session


In [5]:
# Cell 2: Install all dependencies
!pip install -q qdrant-client openai tiktoken gitpython tree-sitter tree-sitter-python tree-sitter-javascript networkx sentence-transformers loguru pydantic pydantic-settings tenacity aiohttp rich typer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 28.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


In [6]:
from pydantic import BaseModel, Field
from typing import List, Dict, Optional, Any, Set
from datetime import datetime
from pathlib import Path
from enum import Enum
from uuid import UUID, uuid4
import os

class CodeLang(str, Enum):
    PYTHON = "python"
    JAVASCRIPT = "javascript"
    TYPESCRIPT = "typescript"
    GO = "go"
    RUST = "rust"
    JAVA = "java"
    UNKNOWN = "unknown"

class RepositoryMetadata(BaseModel):
    url: str
    name: str
    default_branch: str = "main"
    commit_hash: Optional[str] = None
    last_updated: datetime = Field(default_factory=datetime.utcnow)
    total_files: int = 0

class CodeRepository(BaseModel):
    id: UUID = Field(default_factory=uuid4)
    metadata: RepositoryMetadata
    local_path: Path
    status: str = "pending"

class CodeEntity(BaseModel):
    id: UUID = Field(default_factory=uuid4)
    name: str
    entity_type: str
    language: CodeLang
    file_path: Path
    start_line: int
    end_line: int
    documentation: Optional[str] = None
    signature: Optional[str] = None
    body: str
    modifiers: List[str] = Field(default_factory=list)
    calls: Set[str] = Field(default_factory=set)
    imports: Set[str] = Field(default_factory=set)
    parent_class: Optional[str] = None
    namespace: Optional[str] = None

class CodeChunk(BaseModel):
    id: UUID = Field(default_factory=uuid4)
    repository_id: UUID
    file_id: UUID
    file_path: Path
    language: CodeLang
    content: str
    content_hash: str
    start_line: int
    end_line: int
    entity_names: List[str] = Field(default_factory=list)
    chunk_type: str
    embedding: Optional[List[float]] = None
    token_count: int = 0

class RetrievedChunk(BaseModel):
    chunk_id: UUID
    content: str
    file_path: str
    start_line: int
    end_line: int
    language: str
    entity_names: List[str] = Field(default_factory=list)
    combined_score: float = 0.0

class Query(BaseModel):
    id: UUID = Field(default_factory=uuid4)
    text: str
    repository_id: Optional[UUID] = None

class Answer(BaseModel):
    query_id: UUID
    text: str
    confidence_score: float = Field(ge=0.0, le=1.0)
    relevant_files: List[str] = Field(default_factory=list)
    relevant_functions: List[str] = Field(default_factory=list)

In [22]:
import git
import shutil
import hashlib
from pathlib import Path
import fnmatch
import os

class RepositoryManager:
    def __init__(self, base_path="/content/repos"):
        self.base_path = Path(base_path)
        self.base_path.mkdir(parents=True, exist_ok=True)

    def clone(self, url: str, branch: Optional[str] = None):
        repo_name = url.split('/')[-1].replace('.git', '')
        repo_hash = hashlib.sha256(url.encode()).hexdigest()[:8]
        local_path = self.base_path / f"{repo_name}_{repo_hash}"

        if local_path.exists():
            shutil.rmtree(local_path)

        print(f"Cloning {url}...")
        git.Repo.clone_from(url, local_path, branch=branch, depth=1)

        repo = git.Repo(local_path)
        metadata = RepositoryMetadata(
            url=url,
            name=repo_name,
            commit_hash=repo.head.commit.hexsha
        )

        return CodeRepository(
            metadata=metadata,
            local_path=local_path,
            status="cloned"
        )

class FileDiscovery:
    def __init__(self):
        self.ignore_patterns = [
            "**/node_modules/**", "**/.git/**", "**/__pycache__/**",
            "**/*.pyc", "**/.venv/**", "**/dist/**", "**/build/**"
        ]
        self.binary_exts = {'.png', '.jpg', '.pdf', '.zip', '.exe'}

    def discover(self, root_path: Path):
        files = []
        for root, dirs, filenames in os.walk(root_path):
            dirs[:] = [d for d in dirs if not any(
                fnmatch.fnmatch(str(Path(root)/d), p) for p in self.ignore_patterns
            )]

            for fname in filenames:
                fpath = Path(root) / fname
                rel_path = str(fpath.relative_to(root_path))

                if any(fnmatch.fnmatch(rel_path, p) for p in self.ignore_patterns):
                    continue
                if fpath.suffix.lower() in self.binary_exts:
                    continue

                # NEW: Skip docs, tests, licenses, and non-code files
                skip_patterns = ['test', 'tests', 'docs', 'doc', 'LICENSE', 'LICENSE',
                               'README', 'CHANGELOG', 'CONTRIBUTING', 'AUTHORS',
                               '.md', '.rst', '.txt', '.yaml', '.yml', '.json',
                               'Makefile', 'Dockerfile', '.dockerignore', '.gitignore']

                if any(pattern in str(fpath) for pattern in skip_patterns):
                    continue

                size = fpath.stat().st_size
                if size > 5 * 1024 * 1024:
                    continue

                lang = self._detect_lang(fpath.suffix)
                files.append({
                    'path': fpath,
                    'relative': rel_path,
                    'language': lang,
                    'size': size
                })
        return files

    def _detect_lang(self, ext):
        """IMPORTANT: Uses CodeLang NOT Language"""
        mapping = {
            '.py': CodeLang.PYTHON,
            '.js': CodeLang.JAVASCRIPT,
            '.ts': CodeLang.TYPESCRIPT,
            '.go': CodeLang.GO,
            '.rs': CodeLang.RUST,
            '.java': CodeLang.JAVA
        }
        return mapping.get(ext.lower(), CodeLang.UNKNOWN)

    def read_file(self, file_info):
        try:
            with open(file_info['path'], 'r', encoding='utf-8', errors='ignore') as f:
                return f.read()
        except Exception as e:
            print(f"Error reading {file_info['path']}: {e}")
            return ""

In [23]:
!pip install tree sitter language

In [25]:
# Cell 5: Code Parser - FIXED with CodeLang
try:
    from tree_sitter import Language as TSLanguage, Parser
    import tree_sitter_python
    import tree_sitter_javascript
    import tree_sitter_typescript
    import tree_sitter_go
    TREE_SITTER_AVAILABLE = True
except ImportError as e:
    print(f"Tree-sitter import warning: {e}")
    TREE_SITTER_AVAILABLE = False

class CodeParser:
    def __init__(self):
        self.parsers = {}
        if TREE_SITTER_AVAILABLE:
            self._init_parsers()

    def _init_parsers(self):
        """Initialize parsers with correct API"""
        try:
            PY_LANGUAGE = TSLanguage(tree_sitter_python.language())
            self.parsers['python'] = Parser(PY_LANGUAGE)

            JS_LANGUAGE = TSLanguage(tree_sitter_javascript.language())
            self.parsers['javascript'] = Parser(JS_LANGUAGE)

            TS_LANGUAGE = TSLanguage(tree_sitter_typescript.language_typescript())
            self.parsers['typescript'] = Parser(TS_LANGUAGE)

            GO_LANGUAGE = TSLanguage(tree_sitter_go.language())
            self.parsers['go'] = Parser(GO_LANGUAGE)

            print(f"✅ Loaded {len(self.parsers)} parsers")

        except Exception as e:
            print(f"⚠️  Parser init error (will use fallback): {e}")
            self.parsers = {}

    def parse_file(self, file_info, content, repo_id):
        """Extract entities from file"""
        from uuid import uuid4

        file_id = uuid4()
        lang = file_info['language']  # This is now CodeLang type
        lines = content.split('\n')

        entities = []

        # Map CodeLang to parser key
        parser_map = {
            CodeLang.PYTHON: 'python',
            CodeLang.JAVASCRIPT: 'javascript',
            CodeLang.TYPESCRIPT: 'typescript',
            CodeLang.GO: 'go'
        }

        parser_key = parser_map.get(lang)

        if parser_key and parser_key in self.parsers and content:
            try:
                parser = self.parsers[parser_key]
                tree = parser.parse(bytes(content, 'utf8'))
                root = tree.root_node

                def walk(node, depth=0):
                    # Check using CodeLang
                    if lang == CodeLang.PYTHON and node.type in ['function_definition', 'class_definition']:
                        name = self._extract_name(node, content)
                        if name:
                            entities.append(CodeEntity(
                                name=name,
                                entity_type='function' if node.type == 'function_definition' else 'class',
                                language=lang,
                                file_path=file_info['path'],
                                start_line=node.start_point[0] + 1,
                                end_line=node.end_point[0] + 1,
                                body=content[node.start_byte:node.end_byte],
                                signature=content[node.start_byte:node.start_byte+100].split('\n')[0]
                            ))

                    elif lang in [CodeLang.JAVASCRIPT, CodeLang.TYPESCRIPT] and node.type in ['function_declaration', 'class_declaration', 'method_definition']:
                        name = self._extract_name(node, content)
                        if name:
                            entities.append(CodeEntity(
                                name=name,
                                entity_type='function' if 'function' in node.type else 'class',
                                language=lang,
                                file_path=file_info['path'],
                                start_line=node.start_point[0] + 1,
                                end_line=node.end_point[0] + 1,
                                body=content[node.start_byte:node.end_byte],
                                signature=content[node.start_byte:node.start_byte+100].split('\n')[0]
                            ))

                    elif lang == CodeLang.GO and node.type in ['function_declaration', 'method_declaration', 'type_spec']:
                        name = self._extract_name(node, content)
                        if name:
                            entities.append(CodeEntity(
                                name=name,
                                entity_type='function' if 'function' in node.type else 'class',
                                language=lang,
                                file_path=file_info['path'],
                                start_line=node.start_point[0] + 1,
                                end_line=node.end_point[0] + 1,
                                body=content[node.start_byte:node.end_byte],
                                signature=content[node.start_byte:node.start_byte+100].split('\n')[0]
                            ))

                    for child in node.children:
                        walk(child, depth+1)

                walk(root)

            except Exception as e:
                print(f"Parse error for {file_info['path']}: {e}")

        return {
            'id': file_id,
            'repo_id': repo_id,
            'path': file_info['path'],
            'language': lang,
            'content': content,
            'entities': entities,
            'line_count': len(lines)
        }

    def _extract_name(self, node, content):
        """Extract identifier from node"""
        for child in node.children:
            if child.type in ['identifier', 'property_identifier', 'type_identifier']:
                return content[child.start_byte:child.end_byte]
        return None

Tree-sitter import warning: No module named 'tree_sitter_typescript'


In [26]:
# Cell 6: Chunking - FIXED with CodeLang
import tiktoken
import hashlib

class Chunker:
    def __init__(self, chunk_size=512, overlap=128):
        self.encoder = tiktoken.get_encoding("cl100k_base")
        self.chunk_size = chunk_size
        self.overlap = overlap

    def chunk_file(self, parsed_file, repo_id):
        """Create chunks from parsed file"""
        chunks = []
        content = parsed_file['content']
        lang = parsed_file['language']  # CodeLang type

        # If entities found, chunk by entity
        if parsed_file['entities']:
            for entity in parsed_file['entities']:
                chunk = CodeChunk(
                    repository_id=repo_id,
                    file_id=parsed_file['id'],
                    file_path=parsed_file['path'],
                    language=lang,
                    content=entity.body,
                    content_hash=hashlib.sha256(entity.body.encode()).hexdigest(),
                    start_line=entity.start_line,
                    end_line=entity.end_line,
                    entity_names=[entity.name],
                    chunk_type=entity.entity_type,
                    token_count=len(self.encoder.encode(entity.body))
                )
                chunks.append(chunk)
        else:
            # Fallback: sliding window
            tokens = self.encoder.encode(content)
            stride = self.chunk_size - self.overlap

            for i in range(0, len(tokens), stride):
                window = tokens[i:i+self.chunk_size]
                text = self.encoder.decode(window)

                start_char = len(self.encoder.decode(tokens[:i]))
                end_char = start_char + len(text)
                start_line = content[:start_char].count('\n') + 1
                end_line = content[:end_char].count('\n') + 1

                chunk = CodeChunk(
                    repository_id=repo_id,
                    file_id=parsed_file['id'],
                    file_path=parsed_file['path'],
                    language=lang,
                    content=text,
                    content_hash=hashlib.sha256(text.encode()).hexdigest(),
                    start_line=start_line,
                    end_line=end_line,
                    chunk_type='sliding_window',
                    token_count=len(window)
                )
                chunks.append(chunk)

        return chunks

In [27]:
# from sentence_transformers import SentenceTransformer
# import numpy as np

# class EmbeddingGenerator:
#     def __init__(self):
#         print("Loading free embedding model (~400MB)...")
#         # Free, high-quality embeddings
#         self.model = SentenceTransformer('BAAI/bge-large-en-v1.5')
#         self.dimensions = 1024  # This model outputs 1024 dimensions

#     def generate(self, chunks):
#         """Generate embeddings locally (free)"""
#         texts = [c.content for c in chunks]

#         print(f"Encoding {len(texts)} chunks...")
#         embeddings = self.model.encode(
#             texts,
#             batch_size=32,
#             show_progress_bar=True,
#             convert_to_numpy=True
#         )

#         # Attach to chunks
#         for chunk, emb in zip(chunks, embeddings):
#             chunk.embedding = emb.tolist()

#         return chunks

In [28]:
!pip install -q scikit-learn

ERROR: Operation cancelled by user


In [29]:
# Cell 7: TF-IDF with encode method
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

class EmbeddingGenerator:
    def __init__(self):
        print("✅ Using TF-IDF embeddings (no heavy ML libraries)")
        self.vectorizer = TfidfVectorizer(max_features=1024, stop_words='english')
        self.dimensions = 1024
        self.is_fitted = False

    def generate(self, chunks):
        """Generate TF-IDF embeddings (fit on training data)"""
        if not chunks:
            return chunks

        texts = [c.content for c in chunks]

        print(f"Encoding {len(texts)} chunks with TF-IDF...")

        # Fit and transform
        tfidf_matrix = self.vectorizer.fit_transform(texts)
        self.is_fitted = True

        # Convert to dense arrays
        for i, chunk in enumerate(chunks):
            vec = tfidf_matrix[i].toarray()[0]
            chunk.embedding = self._pad_and_normalize(vec)

        print(f"✅ Encoded {len(chunks)} chunks")
        return chunks

    def encode(self, texts):
        """Encode new texts (for queries) - mimics sentence_transformers API"""
        if isinstance(texts, str):
            texts = [texts]

        if not self.is_fitted:
            # If not fitted yet, return random/zero vector (shouldn't happen in normal flow)
            return [np.zeros(self.dimensions).tolist() for _ in texts]

        # Transform (don't fit) new text
        tfidf_matrix = self.vectorizer.transform(texts)

        embeddings = []
        for i in range(len(texts)):
            vec = tfidf_matrix[i].toarray()[0]
            embeddings.append(self._pad_and_normalize(vec))

        return np.array(embeddings)

    def _pad_and_normalize(self, vec):
        """Pad/truncate and normalize vector"""
        # Ensure exactly 1024 dimensions
        if len(vec) < 1024:
            vec = np.pad(vec, (0, 1024 - len(vec)))
        else:
            vec = vec[:1024]

        # L2 normalize
        norm = np.linalg.norm(vec)
        if norm > 0:
            vec = vec / norm

        return vec.tolist()

In [30]:
# Cell 8: Vector Store - FIXED (with divide by zero protection)
import numpy as np
from typing import List, Dict, Optional

class SimpleVectorStore:
    """Simple in-memory vector store using NumPy"""
    def __init__(self):
        self.vectors = []
        self.collection_name = "codebase_chunks"

    def initialize(self):
        print("Using simple in-memory vector store")

    def upsert(self, chunks):
        for chunk in chunks:
            if chunk.embedding:
                self.vectors.append({
                    'id': str(chunk.id),
                    'vector': np.array(chunk.embedding),
                    'payload': {
                        "repo_id": str(chunk.repository_id),
                        "file_path": str(chunk.file_path),
                        "content": chunk.content,
                        "start_line": chunk.start_line,
                        "end_line": chunk.end_line,
                        "language": str(chunk.language),
                        "entity_names": chunk.entity_names,
                    }
                })
        print(f"Stored {len(chunks)} chunks in memory")

    def search(self, query_vector, repo_id=None, top_k=10):
        """Cosine similarity search with divide-by-zero protection"""
        if not self.vectors:
            return []

        query = np.array(query_vector)
        query_norm = np.linalg.norm(query)
        results = []

        for item in self.vectors:
            # Filter by repo_id if specified
            if repo_id and item['payload']['repo_id'] != str(repo_id):
                continue

            # Cosine similarity with safety check
            vec = item['vector']
            vec_norm = np.linalg.norm(vec)

            # FIX: Check for zero norms to avoid division by zero
            if query_norm > 0 and vec_norm > 0:
                similarity = np.dot(query, vec) / (query_norm * vec_norm)
            else:
                similarity = 0.0

            results.append({
                'id': item['id'],
                'score': float(similarity),
                'payload': item['payload']
            })

        # Sort by score
        results.sort(key=lambda x: x['score'], reverse=True)

        # Convert to RetrievedChunk
        chunks = []
        for res in results[:top_k]:
            p = res['payload']
            chunks.append(RetrievedChunk(
                chunk_id=res['id'],
                content=p['content'],
                file_path=p['file_path'],
                start_line=p['start_line'],
                end_line=p['end_line'],
                language=p['language'],
                entity_names=p.get('entity_names', []),
                combined_score=res['score']
            ))
        return chunks

# Use SimpleVectorStore directly (Qdrant optional)
VectorStore = SimpleVectorStore

In [31]:
# Cell 9: Groq API - UPDATED MODEL
from openai import OpenAI
from google.colab import userdata
import numpy as np

class GroqGenerator:
    def __init__(self):
        self.client = OpenAI(
            api_key=userdata.get('GROQ_API_KEY'),
            base_url="https://api.groq.com/openai/v1"
        )
        # Updated model - llama3-8b-8192 is deprecated
        self.model = "llama-3.1-8b-instant"  # or "mixtral-8x7b-32768" or "gemma2-9b-it"

    def generate(self, question, chunks):
        """Generate answer using Groq"""
        context_parts = []
        for i, chunk in enumerate(chunks[:5], 1):
            context_parts.append(
                f"[{i}] File: {chunk.file_path} (Lines {chunk.start_line}-{chunk.end_line})\n"
                f"```{chunk.language}\n{chunk.content[:1000]}\n```"
            )

        context = "\n\n".join(context_parts)

        system_prompt = """You are an expert code analyst. Answer questions about the provided code.
Always cite specific files and line numbers.
If you're unsure, say so and provide a confidence score (0.0-1.0)."""

        user_prompt = f"""Context code:
{context}

Question: {question}

Provide a detailed answer with citations like [File: path/to/file.py:10-20]."""

        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.1,
                max_tokens=2048
            )

            answer_text = response.choices[0].message.content

            # Calculate confidence based on retrieval scores
            avg_score = sum(c.combined_score for c in chunks) / len(chunks) if chunks else 0
            confidence = min(avg_score * 1.2, 0.95)

            return Answer(
                query_id=uuid4(),
                text=answer_text,
                confidence_score=confidence,
                relevant_files=list(set(c.file_path for c in chunks)),
                relevant_functions=list(set(
                    name for c in chunks for name in c.entity_names
                ))
            )

        except Exception as e:
            print(f"Groq API error: {e}")
            return Answer(
                query_id=uuid4(),
                text=f"Error: {str(e)}",
                confidence_score=0.0,
                relevant_files=[]
            )

In [32]:
# Cell 10: Main Pipeline - FIXED SYNTAX
class CodebaseQA:
    def __init__(self):
        self.repo_manager = RepositoryManager()
        self.discovery = FileDiscovery()
        self.parser = CodeParser()
        self.chunker = Chunker(chunk_size=512)
        self.embedder = EmbeddingGenerator()
        # Use SimpleVectorStore (reliable, no dependencies)
        self.vector_store = SimpleVectorStore()
        self.llm = GroqGenerator()
        self.current_repo = None

    def ingest(self, github_url):
        """Full ingestion pipeline"""
        print("🚀 Starting ingestion...")

        # 1. Clone
        repo = self.repo_manager.clone(github_url)
        self.current_repo = repo
        print(f"✅ Cloned: {repo.metadata.name}")

        # 2. Initialize vector store
        self.vector_store.initialize()

        # 3. Discover files
        files = self.discovery.discover(repo.local_path)
        print(f"📁 Found {len(files)} files")

        if len(files) > 200:
            print(f"⚠️  Large repo! Processing first 200 files only...")
            files = files[:200]

        # 4. Parse and chunk
        all_chunks = []
        for i, file_info in enumerate(files):
            if i % 20 == 0:
                print(f"   Processing {i}/{len(files)}...")

            content = self.discovery.read_file(file_info)
            if not content or len(content) > 100000:
                continue

            parsed = self.parser.parse_file(file_info, content, repo.id)
            chunks = self.chunker.chunk_file(parsed, repo.id)
            all_chunks.extend(chunks)

        print(f"✂️  Created {len(all_chunks)} chunks")

        if not all_chunks:
            print("❌ No chunks created!")
            return None

        # 5. Generate embeddings
        print("🧠 Generating embeddings...")
        all_chunks = self.embedder.generate(all_chunks)

        # 6. Store
        print("💾 Storing in vector database...")
        self.vector_store.upsert(all_chunks)

        repo.status = "indexed"
        print("✅ Ingestion complete!")
        return repo.id

    def ask(self, question):
        """Query the codebase"""
        if not self.current_repo:
            print("❌ No repository ingested yet! Run .ingest() first")
            return None

        print(f"🔍 Searching for: {question}")

        # Handle TF-IDF embedding
        q_vec = self.embedder.vectorizer.transform([question]).toarray()[0]

        # Pad to 1024 dimensions
        if len(q_vec) < 1024:
            q_vec = np.pad(q_vec, (0, 1024 - len(q_vec)))
        else:
            q_vec = q_vec[:1024]

        # Normalize
        norm = np.linalg.norm(q_vec)
        if norm > 0:
            q_vec = q_vec / norm

        q_embedding = q_vec.tolist()

        # 2. Retrieve
        chunks = self.vector_store.search(
            q_embedding,
            repo_id=self.current_repo.id,
            top_k=5
        )

        if not chunks:
            print("❌ No relevant code found")
            return None

        print(f"📋 Retrieved {len(chunks)} relevant chunks")

        # 3. Generate answer
        print("🤖 Generating answer with Groq...")
        answer = self.llm.generate(question, chunks)

        return answer, chunks

In [34]:
# Cell 11: Run the system
from google.colab import userdata
import os

# Set API key (add to Secrets first!)
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

# Initialize
qa = CodebaseQA()

# Ingest repository
repo_id = qa.ingest("https://github.com/psf/requests")

# Test multiple questions
questions = [
    "How does the authentication work?",
    "How do I send a POST request with JSON?",
    "Show me the session handling code",
    "Where are the timeout settings?"
]

for question in questions:
    print(f"\n{'='*60}")
    print(f"❓ QUESTION: {question}")
    print(f"{'='*60}")

    answer, chunks = qa.ask(question)

    print(f"\n🤖 ANSWER (Confidence: {answer.confidence_score:.2f})")
    print(f"{'='*60}")
    print(answer.text)
    print(f"\n📁 RELEVANT FILES:")
    for f in set(answer.relevant_files[:3]):
        print(f"   • {f}")

    print("\n" + "="*60)

✅ Using TF-IDF embeddings (no heavy ML libraries)
🚀 Starting ingestion...
Cloning https://github.com/psf/requests...
✅ Cloned: requests
Using simple in-memory vector store
📁 Found 36 files
   Processing 0/36...
   Processing 20/36...
✂️  Created 172 chunks
🧠 Generating embeddings...
Encoding 172 chunks with TF-IDF...
✅ Encoded 172 chunks
💾 Storing in vector database...
Stored 172 chunks in memory
✅ Ingestion complete!

❓ QUESTION: How does the authentication work?
🔍 Searching for: How does the authentication work?
📋 Retrieved 5 relevant chunks
🤖 Generating answer with Groq...

🤖 ANSWER (Confidence: 0.07)
The authentication process in the Requests library is handled by the `rebuild_auth` method in the `requests/sessions.py` file [1:305-366]. This method is responsible for intelligently removing and reaplying authentication headers when a redirect occurs.

Here's a step-by-step explanation of how authentication works:

1. **Original Request**: When a request is made, the `rebuild_auth` m